# MLflow Example: Tracking Two Model Training Runs

In [2]:
# === Model Training and Experimentation ===

# Import necessary libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits


In [3]:

# Load dataset
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

# Define a function to train a model and return its accuracy
def train_model(n_estimators, max_depth):
    # Train a RandomForestClassifier
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    print(f"Model with n_estimators={n_estimators}, max_depth={max_depth} achieved accuracy={accuracy:.4f}")
    return model, accuracy

In [4]:

# Train models with different hyperparameter configurations
print("Training Model 1...")
model_1, acc_1 = train_model(n_estimators=20, max_depth=5)

Training Model 1...
Model with n_estimators=20, max_depth=5 achieved accuracy=0.9389


In [5]:
print("\nTraining Model 2...")
model_2, acc_2 = train_model(n_estimators=100, max_depth=10)



Training Model 2...
Model with n_estimators=100, max_depth=10 achieved accuracy=0.9722


Now adding MLFlow

In [6]:
! pip3 install mlflow

In [7]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_digits

In [8]:
!pip3 install dagshub

In [9]:
# get this code from dagshub
import dagshub
dagshub.init(repo_owner='zorobabel33', repo_name='dvc', mlflow=True)



Initialized MLflow to track repo "zorobabel33/dvc"

Repository zorobabel33/dvc initialized!

In [17]:
data = load_digits()
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.2, random_state=42)

In [18]:
# Set MLflow experiment name. This will get created if it doesn't exist
experiment_name = "RandomForest_V1.0.1"
mlflow.set_experiment(experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/c1a4586c54144f7994d3129662e0cc09', creation_time=1794846136030, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1794846136030, lifecycle_stage='active', name='RandomForest_V1.0.1', tags={}, trace_location=None, workspace='default'>

In [19]:
# Define a function to train a model, log parameters and metrics to MLflow
def train_and_log_model(n_estimators, max_depth, run_name):
    with mlflow.start_run(): # <--
        # Train a RandomForestClassifier
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        # Log parameters, metrics, and the model itself
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(
            model,
            run_name,
            skops_trusted_types=["sklearn.tree._tree.Tree"],
            registered_model_name="digits-rf",
        )



        print(f"Logged RandomForest model with n_estimators={n_estimators}, max_depth={max_depth}, accuracy={accuracy:.4f}")


In [20]:
# Train and log Model 1
print("Training and Logging Model 1...")
train_and_log_model(n_estimators=20, max_depth=5, run_name="random_forest_model_run_1")


Training and Logging Model 1...


2026/11/16 17:24:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'digits-rf' already exists. Creating a new version of this model...
2026/11/16 17:24:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 3
Created version '3' of model 'digits-rf'.


Logged RandomForest model with n_estimators=20, max_depth=5, accuracy=0.9389
🏃 View run mysterious-shoat-376 at: https://dagshub.com/zorobabel33/dvc.mlflow/#/experiments/0/runs/20c25230f9b04554841b055dd51e623f
🧪 View experiment at: https://dagshub.com/zorobabel33/dvc.mlflow/#/experiments/0


In [21]:
# Train and log Model 2
print("\nTraining and Logging Model 2...")
train_and_log_model(n_estimators=100, max_depth=10, run_name="random_forest_model_run_V1_0_2")



Training and Logging Model 2...


2026/11/16 17:25:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'digits-rf' already exists. Creating a new version of this model...
2026/11/16 17:26:03 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: digits-rf, version 4
Created version '4' of model 'digits-rf'.


Logged RandomForest model with n_estimators=100, max_depth=10, accuracy=0.9722
🏃 View run bright-midge-697 at: https://dagshub.com/zorobabel33/dvc.mlflow/#/experiments/0/runs/2380516dd9264306b81245c87e55fb0b
🧪 View experiment at: https://dagshub.com/zorobabel33/dvc.mlflow/#/experiments/0


In [15]:
# Instructions to visualize results
print("\nTo view the results, run the following command in your terminal:")
print("mlflow ui")
print("Then navigate to http://127.0.0.1:5000 to explore the experiment results.")


To view the results, run the following command in your terminal:
mlflow ui
Then navigate to http://127.0.0.1:5000 to explore the experiment results.
